# Understanding Quantization and Low-Rank Adaptation (LoRA)

## Part 1: Quantization

#Quantization is a technique used to reduce the memory footprint and computational requirements of machine learning models. It works by representing weights and activations with lower precision data types.



In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Original high-precision weights (32-bit floating point)
original_weights = np.random.normal(0, 1, 1000)

# Function to perform simple linear quantization
def linear_quantize(values, bits=8):
    """
    Quantize floating point values to specified number of bits
    """
    # Calculate the range of values
    min_val, max_val = np.min(values), np.max(values)

    # Calculate the step size (distance between quantization levels)
    step_size = (max_val - min_val) / (2**bits - 1)

    # Quantize the values to integers
    quantized_ints = np.round((values - min_val) / step_size).astype(int)

    # Convert back to floating point using the same scale
    quantized_values = quantized_ints * step_size + min_val

    return quantized_values, quantized_ints, min_val, max_val, step_size

# Perform 8-bit quantization
quantized_weights_8bit, quantized_ints_8bit, min_val, max_val, step_size = linear_quantize(original_weights, bits=8)

# Perform 4-bit quantization for comparison
quantized_weights_4bit, quantized_ints_4bit, _, _, _ = linear_quantize(original_weights, bits=4)

# Calculate quantization error
error_8bit = np.abs(original_weights - quantized_weights_8bit)
error_4bit = np.abs(original_weights - quantized_weights_4bit)

# Plot results
plt.figure(figsize=(15, 10))

# Original vs 8-bit quantized weights
plt.subplot(2, 2, 1)
plt.scatter(original_weights[:100], quantized_weights_8bit[:100], alpha=0.5)
plt.plot([min_val, max_val], [min_val, max_val], 'r--')
plt.title('Original vs 8-bit Quantized Weights')
plt.xlabel('Original Weights')
plt.ylabel('Quantized Weights')

# Distribution of error for 8-bit quantization
plt.subplot(2, 2, 2)
plt.hist(error_8bit, bins=50)
plt.title('8-bit Quantization Error')
plt.xlabel('Absolute Error')
plt.ylabel('Frequency')

# Original vs 4-bit quantized weights
plt.subplot(2, 2, 3)
plt.scatter(original_weights[:100], quantized_weights_4bit[:100], alpha=0.5)
plt.plot([min_val, max_val], [min_val, max_val], 'r--')
plt.title('Original vs 4-bit Quantized Weights')
plt.xlabel('Original Weights')
plt.ylabel('Quantized Weights')

# Distribution of error for 4-bit quantization
plt.subplot(2, 2, 4)
plt.hist(error_4bit, bins=50)
plt.title('4-bit Quantization Error')
plt.xlabel('Absolute Error')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

print(f"Original data size (32-bit): {original_weights.nbytes} bytes")
print(f"8-bit quantized integers size: {quantized_ints_8bit.nbytes} bytes")
print(f"4-bit quantized data size (estimated): {original_weights.size * 4/8} bytes")
print(f"Average error with 8-bit quantization: {np.mean(error_8bit):.6f}")
print(f"Average error with 4-bit quantization: {np.mean(error_4bit):.6f}")


In [ ]:
#Different types of quantization
def uniform_quantize(values, bits=8):
    """
    Uniform quantization: evenly spaced quantization levels
    """
    min_val, max_val = np.min(values), np.max(values)
    step_size = (max_val - min_val) / (2**bits - 1)

    quantized_ints = np.round((values - min_val) / step_size).astype(int)
    quantized_values = quantized_ints * step_size + min_val

    return quantized_values, quantized_ints

def symmetric_quantize(values, bits=8):
    """
    Symmetric quantization: zero is exactly represented, and range is symmetric
    """
    # Find the maximum absolute value
    max_abs = np.max(np.abs(values))

    # Define the quantization step size
    step_size = (2 * max_abs) / (2**bits - 1)

    # Quantize the values
    quantized_ints = np.round(values / step_size).astype(int)
    quantized_values = quantized_ints * step_size

    return quantized_values, quantized_ints

def log_quantize(values, bits=8):
    """
    Logarithmic quantization: more precision near zero
    """
    # Add a small epsilon to avoid log(0)
    epsilon = 1e-10

    # Determine sign and take log of absolute values
    signs = np.sign(values)
    log_abs = np.log(np.abs(values) + epsilon)

    # Normalize to [0, 1]
    min_log, max_log = np.min(log_abs), np.max(log_abs)
    normalized_log = (log_abs - min_log) / (max_log - min_log)

    # Quantize the normalized log values
    levels = 2**bits - 1
    quantized_ints = np.round(normalized_log * levels).astype(int)

    # Convert back to the original scale
    denorm_log = quantized_ints / levels * (max_log - min_log) + min_log
    quantized_values = signs * np.exp(denorm_log)

    return quantized_values, quantized_ints

# Generate some weights with a range of values to illustrate differences
weights = np.concatenate([
    np.random.normal(0, 0.01, 500),  # small values
    np.random.normal(0, 1, 500)       # larger values
])

# Apply different quantization schemes
uniform_quant, _ = uniform_quantize(weights, bits=4)
symmetric_quant, _ = symmetric_quantize(weights, bits=4)
log_quant, _ = log_quantize(weights, bits=4)

# Plot results
plt.figure(figsize=(15, 10))

# Original vs uniform quantization
plt.subplot(2, 2, 1)
plt.scatter(weights, uniform_quant, alpha=0.5)
plt.plot([np.min(weights), np.max(weights)], [np.min(weights), np.max(weights)], 'r--')
plt.title('Uniform Quantization (4-bit)')
plt.xlabel('Original Weights')
plt.ylabel('Quantized Weights')

# Original vs symmetric quantization
plt.subplot(2, 2, 2)
plt.scatter(weights, symmetric_quant, alpha=0.5)
plt.plot([np.min(weights), np.max(weights)], [np.min(weights), np.max(weights)], 'r--')
plt.title('Symmetric Quantization (4-bit)')
plt.xlabel('Original Weights')
plt.ylabel('Quantized Weights')

# Original vs logarithmic quantization
plt.subplot(2, 2, 3)
plt.scatter(weights, log_quant, alpha=0.5)
plt.plot([np.min(weights), np.max(weights)], [np.min(weights), np.max(weights)], 'r--')
plt.title('Logarithmic Quantization (4-bit)')
plt.xlabel('Original Weights')
plt.ylabel('Quantized Weights')

# Compare the three methods for small values
plt.subplot(2, 2, 4)
small_idx = np.abs(weights) < 0.05
plt.scatter(weights[small_idx], uniform_quant[small_idx], alpha=0.5, label='Uniform')
plt.scatter(weights[small_idx], symmetric_quant[small_idx], alpha=0.5, label='Symmetric')
plt.scatter(weights[small_idx], log_quant[small_idx], alpha=0.5, label='Logarithmic')
plt.plot([np.min(weights[small_idx]), np.max(weights[small_idx])],
         [np.min(weights[small_idx]), np.max(weights[small_idx])], 'r--')
plt.title('Comparison for Small Values')
plt.xlabel('Original Weights')
plt.ylabel('Quantized Weights')
plt.legend()

plt.tight_layout()
plt.show()



### 1.3 Implementing Per-Channel Quantization

Now let's implement per-channel quantization, which is commonly used in neural networks:



In [ ]:

def per_channel_quantize(weight_tensor, bits=8, axis=0):
    """
    Perform per-channel quantization on a weight tensor

    Args:
        weight_tensor: Input tensor with shape (out_channels, in_channels, ...)
        bits: Number of bits for quantization
        axis: Channel axis (usually 0 for weights)

    Returns:
        Quantized tensor and quantization parameters
    """
    # Store original shape
    original_shape = weight_tensor.shape

    # Get number of channels
    num_channels = original_shape[axis]

    # Reshape to separate the channel dimension
    reshaped = np.moveaxis(weight_tensor, axis, 0)
    reshaped_shape = reshaped.shape
    flattened = reshaped.reshape(num_channels, -1)

    # Initialize outputs
    quantized = np.zeros_like(flattened)
    scales = np.zeros(num_channels)
    zero_points = np.zeros(num_channels, dtype=np.int32)

    # Quantize each channel separately
    for c in range(num_channels):
        # Get min and max values for this channel
        min_val = np.min(flattened[c])
        max_val = np.max(flattened[c])

        # Calculate scale and zero point
        scale = (max_val - min_val) / (2**bits - 1)

        # Avoid division by zero
        if scale == 0:
            scale = 1.0

        zero_point = -round(min_val / scale)

        # Clip zero point to valid range
        zero_point = max(0, min(2**bits - 1, zero_point))

        # Quantize the channel
        quantized[c] = np.clip(np.round(flattened[c] / scale) + zero_point, 0, 2**bits - 1)

        # Store quantization parameters
        scales[c] = scale
        zero_points[c] = zero_point

    # Reshape back to original shape
    quantized = quantized.reshape(reshaped_shape)
    quantized = np.moveaxis(quantized, 0, axis)

    return quantized, scales, zero_points

def dequantize(quantized_tensor, scales, zero_points, axis=0):
    """
    Dequantize a tensor that was quantized per-channel
    """
    # Store original shape
    original_shape = quantized_tensor.shape

    # Reshape to separate the channel dimension
    reshaped = np.moveaxis(quantized_tensor, axis, 0)
    reshaped_shape = reshaped.shape
    flattened = reshaped.reshape(reshaped_shape[0], -1)

    # Initialize output
    dequantized = np.zeros_like(flattened, dtype=np.float32)

    # Dequantize each channel
    for c in range(reshaped_shape[0]):
        dequantized[c] = scales[c] * (flattened[c] - zero_points[c])

    # Reshape back to original shape
    dequantized = dequantized.reshape(reshaped_shape)
    dequantized = np.moveaxis(dequantized, 0, axis)

    return dequantized

# Create a mock convolutional layer weights
conv_weights = np.random.normal(0, 1, (16, 3, 3, 3))  # 16 output channels, 3 input channels, 3x3 kernel

# Quantize the weights
quantized_weights, scales, zero_points = per_channel_quantize(conv_weights, bits=8)

# Dequantize to see how close we get to the original
dequantized_weights = dequantize(quantized_weights, scales, zero_points)

# Calculate the error
quantization_error = np.abs(conv_weights - dequantized_weights)

print(f"Original weights shape: {conv_weights.shape}")
print(f"Quantized weights shape: {quantized_weights.shape}")
print(f"Quantization scales shape: {scales.shape}")
print(f"Average quantization error: {np.mean(quantization_error):.6f}")
print(f"Max quantization error: {np.max(quantization_error):.6f}")

# Plot the results for the first channel
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(conv_weights[0, 0], cmap='viridis')
plt.title('Original Weights (Channel 0, 0)')
plt.colorbar()

plt.subplot(1, 3, 2)
plt.imshow(dequantized_weights[0, 0], cmap='viridis')
plt.title('Dequantized Weights')
plt.colorbar()

plt.subplot(1, 3, 3)
plt.imshow(quantization_error[0, 0], cmap='hot')
plt.title('Quantization Error')
plt.colorbar()

plt.tight_layout()
plt.show()



### 1.4 Implementing a Simple Neural Network with Quantization

Let's see how quantization affects the performance of a simple neural network:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Define a simple CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7)
        x = self.fc(x)
        return x

# Function to quantize a PyTorch model
def quantize_model(model, bits=8):
    """
    Quantize the weights of a PyTorch model (in-place)
    Returns a dictionary of quantization parameters
    """
    quant_params = {}

    for name, param in model.named_parameters():
        if 'weight' in name:
            # Get the tensor as numpy array
            weight_np = param.data.cpu().numpy()

            # Determine the appropriate axis for per-channel quantization
            axis = 0  # Default for linear layers
            if len(weight_np.shape) == 4:  # Conv layers
                axis = 0  # Output channels

            # Quantize the weights
            quantized, scales, zero_points = per_channel_quantize(weight_np, bits=bits, axis=axis)

            # Store the quantization parameters
            quant_params[name] = {
                'scales': scales,
                'zero_points': zero_points,
                'bits': bits,
                'axis': axis
            }

            # Convert back to torch tensor and update the model
            param.data = torch.from_numpy(quantized).to(param.device)

    return quant_params

def dequantize_model(model, quant_params):
    """
    Dequantize the weights of a PyTorch model (in-place)
    """
    for name, param in model.named_parameters():
        if name in quant_params and 'weight' in name:
            # Get the parameters
            params = quant_params[name]

            # Get the tensor as numpy array
            weight_np = param.data.cpu().numpy()

            # Dequantize the weights
            dequantized = dequantize(
                weight_np,
                params['scales'],
                params['zero_points'],
                axis=params['axis']
            )

            # Convert back to torch tensor and update the model
            param.data = torch.from_numpy(dequantized).to(param.device)

# Define a function to test the model
def test_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data[0].to(device), data[1].to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

# Set up data loaders
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])


In [ ]:

# Load MNIST dataset (this will download if it doesn't exist)
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# Train the model
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train for just a few epochs for demonstration
num_epochs = 2
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 100 == 99:
            print(f'[{epoch + 1}, {i + 1}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0

print('Finished Training')

# Test the original model
original_accuracy = test_model(model, test_loader, device)
print(f"Original model accuracy: {original_accuracy:.2f}%")

# Save the original model weights
original_weights = {name: param.data.clone() for name, param in model.named_parameters()}

# Quantize the model with different bit widths and measure accuracy
bit_widths = [8, 4, 2]
results = []

for bits in bit_widths:
    # Restore the original weights
    for name, param in model.named_parameters():
        param.data = original_weights[name].clone()

    # Quantize the model
    quant_params = quantize_model(model, bits=bits)

    # Dequantize for inference
    dequantize_model(model, quant_params)

    # Test the quantized model
    quantized_accuracy = test_model(model, test_loader, device)

    # Calculate model size
    original_size = sum(p.numel() * 4 for p in model.parameters())  # 4 bytes for float32
    quantized_size = sum(p.numel() * bits / 8 for p in model.parameters())

    # Store results
    results.append((bits, quantized_accuracy, original_size, quantized_size))

    print(f"{bits}-bit quantized model accuracy: {quantized_accuracy:.2f}%")
    print(f"Compression ratio: {original_size / quantized_size:.2f}x")

# Restore the original weights
for name, param in model.named_parameters():
    param.data = original_weights[name].clone()

# Plot results
plt.figure(figsize=(10, 6))
bits, accuracies, _, _ = zip(*results)
plt.plot(bits, accuracies, marker='o')
plt.axhline(y=original_accuracy, color='r', linestyle='--', label=f'Original (32-bit): {original_accuracy:.2f}%')
plt.xlabel('Bit Width')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy vs. Quantization Bit Width')
plt.grid(True)
plt.legend()
plt.show()


## Part 2: Low-Rank Adaptation (LoRA)
### 2.1 Understanding Low-Rank Decomposition


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

# Create a random matrix
np.random.seed(42)
original_matrix = np.random.normal(0, 1, (100, 100))

# Add some structure to make it low-rank
for i in range(10):
    vector = np.random.normal(0, 5, 100)
    original_matrix += np.outer(vector, vector)

# Function to perform SVD and low-rank approximation
def low_rank_approximation(matrix, rank):
    """
    Decompose a matrix into its low-rank approximation using SVD
    """
    # Perform SVD
    U, S, Vt = np.linalg.svd(matrix, full_matrices=False)

    # Use only the top-k singular values and vectors
    U_k = U[:, :rank]
    S_k = np.diag(S[:rank])
    Vt_k = Vt[:rank, :]

    # Reconstruct the matrix
    reconstructed = U_k @ S_k @ Vt_k

    return reconstructed, U_k, S_k, Vt_k, S

# Calculate approximations for different ranks
ranks = [1, 5, 10, 20, 50]
approximations = []
singular_values = None

for rank in ranks:
    approx, U_k, S_k, Vt_k, S = low_rank_approximation(original_matrix, rank)
    approximations.append(approx)
    if singular_values is None:
        singular_values = S

# Calculate the error for each approximation
errors = [np.linalg.norm(original_matrix - approx, 'fro') for approx in approximations]
relative_errors = [error / np.linalg.norm(original_matrix, 'fro') for error in errors]

# Plot the singular values
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.semilogy(singular_values, 'o-')
plt.title('Singular Values')
plt.xlabel('Index')
plt.ylabel('Magnitude (log scale)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(ranks, relative_errors, 'o-')
plt.title('Relative Approximation Error')
plt.xlabel('Rank')
plt.ylabel('||A - A_k||_F / ||A||_F')
plt.grid(True)

plt.tight_layout()
plt.show()

# Visualize the original matrix and its approximations
plt.figure(figsize=(15, 8))

plt.subplot(2, 3, 1)
plt.imshow(original_matrix, cmap='viridis')
plt.title('Original Matrix')
plt.colorbar()

for i, rank in enumerate(ranks[:5]):  # Show only first 5 approximations
    plt.subplot(2, 3, i + 2)
    plt.imshow(approximations[i], cmap='viridis')
    plt.title(f'Rank-{rank} Approximation')
    plt.colorbar()

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class LoRALayer(nn.Module):
    """
    Implementation of a LoRA (Low-Rank Adaptation) layer
    """
    def __init__(self, in_features, out_features, rank=4, alpha=1.0):
        super(LoRALayer, self).__init__()

        # Original weight matrix (frozen)
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))

        # Low-rank matrices for adaptation
        # Note: lora_A has shape (in_features, rank) and lora_B has shape (rank, out_features)
        self.lora_A = nn.Parameter(torch.Tensor(in_features, rank))
        self.lora_B = nn.Parameter(torch.Tensor(rank, out_features))

        # Scaling factor
        self.alpha = alpha
        self.rank = rank

        # Bias term
        self.bias = nn.Parameter(torch.Tensor(out_features))

        # Initialize weights
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)  # Initialize B to zero so LoRA starts as identity
        nn.init.zeros_(self.bias)

    def forward(self, x):
        # Original forward pass - F.linear expects weights as (out_features, in_features)
        original_output = F.linear(x, self.weight, self.bias)

        # LoRA adaptation path: x @ A @ B
        # First multiply x by A: (batch_size, in_features) @ (in_features, rank) = (batch_size, rank)
        lora_output = x @ self.lora_A
        # Then multiply by B: (batch_size, rank) @ (rank, out_features) = (batch_size, out_features)
        lora_output = lora_output @ self.lora_B

        # Apply scaling factor to LoRA output and add to original output
        return original_output + (self.alpha / self.rank) * lora_output

# Let's create a simple LoRA-enabled MLP for MNIST
class LoRAMLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, output_dim=10, rank=4, alpha=1.0):
        super(LoRAMLP, self).__init__()

        # Define layers
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = LoRALayer(hidden_dim, hidden_dim, rank=rank, alpha=alpha)
        self.fc3 = LoRALayer(hidden_dim, output_dim, rank=rank, alpha=alpha)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 784)  # Flatten the input
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x


In [ ]:

# Function to count trainable parameters
def count_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    return trainable_params, total_params

# Training function for the models
def train_model(model, train_loader, test_loader, optimizer, criterion, epochs=3, device='cpu'):
    model.to(device)
    train_losses = []
    test_accuracies = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f'Epoch: {epoch+1}/{epochs} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
                      f'({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')

        # Evaluate on test set
        model.eval()
        correct = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        accuracy = 100. * correct / len(test_loader.dataset)
        print(f'Epoch: {epoch+1}/{epochs}, Test Accuracy: {accuracy:.2f}%')

        train_losses.append(running_loss / len(train_loader))
        test_accuracies.append(accuracy)

    return train_losses, test_accuracies

# Example usage:
# Create models for comparison
lora_model = LoRAMLP(rank=8, alpha=16.0)

# Make the original weights non-trainable for LoRA fine-tuning
for name, param in lora_model.named_parameters():
    if 'lora' not in name:  # Freeze all non-LoRA parameters
        param.requires_grad = False

lora_trainable, lora_total = count_parameters(lora_model)
print(f"LoRA model trainable parameters: {lora_trainable:,} ({lora_trainable/lora_total:.2%} of total)")

# Regular model for comparison (no LoRA)
regular_model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)
regular_trainable, regular_total = count_parameters(regular_model)
print(f"Regular model trainable parameters: {regular_trainable:,}")

In [ ]:
# Training function
def train_model(model, train_loader, test_loader, optimizer, criterion, epochs=5, device='cpu'):
    model.to(device)
    train_losses = []
    test_accuracies = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            if batch_idx % 100 == 99:
                print(f'Epoch: {epoch+1}/{epochs} [{batch_idx+1}/{len(train_loader)}] Loss: {running_loss/100:.4f}')
                train_losses.append(running_loss/100)
                running_loss = 0.0

        # Evaluate on test set
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                outputs = model(data)
                _, predicted = torch.max(outputs.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

        accuracy = 100 * correct / total
        test_accuracies.append(accuracy)
        print(f'Epoch: {epoch+1}/{epochs}, Test Accuracy: {accuracy:.2f}%')

    return train_losses, test_accuracies

In [ ]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Set up the device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set up data loaders
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# Set up optimizers and criterion
criterion = nn.CrossEntropyLoss()
lora_optimizer = torch.optim.Adam([p for p in lora_model.parameters() if p.requires_grad], lr=0.001)
regular_optimizer = torch.optim.Adam(regular_model.parameters(), lr=0.001)

# Train LoRA model
print("Training LoRA model...")
lora_train_losses, lora_test_accuracies = train_model(
    lora_model, train_loader, test_loader, lora_optimizer, criterion, epochs=3, device=device
)



